# RAG Pipeline with RRF Fusion for Omnilex Legal Retrieval

## Overview
This notebook implements a **Retrieval-Augmented Generation (RAG) pipeline** with **Reciprocal Rank Fusion (RRF)** for Swiss legal citation retrieval. It combines multiple retrieval methods (BM25 keyword search + Dense semantic search) using RRF to improve citation ranking.

## Hardware Optimization
- **GPU**: Nvidia T4 (15GB VRAM) - Used for sentence-transformers embeddings
- **RAM**: 12GB - Memory-efficient chunked loading and processing
- **FAISS**: Uses faiss-cpu (GPU acceleration via torch for embeddings)

## Pipeline Steps
1. Setup & Install Dependencies
2. Configure GPU and Memory Settings
3. Load Query & Legal Corpora Data (Memory-Efficient)
4. Build/Load Retrieval Indices (BM25 + Dense)
5. Retrieve & Fuse Results with RRF
6. Evaluate on Validation Set
7. Generate Submission File

## Extendability
- Add new retrievers (e.g., TF-IDF, custom models) in Section 5
- Modify RRF parameters or add new fusion methods
- Integrate LLM reranking or citation generation
- Swap embedding models or BM25 tokenizers

## Note
Dataset files are not included in the repo but will be provided later. Use the data upload/mount cells below to load your datasets.

## 1. Setup & Install Dependencies

In [1]:
# Install required packages (optimized for Nvidia T4 GPU - 15GB VRAM, 12GB RAM)
!pip install -q git+https://github.com/samin-irtiza/Omnilex-Agentic-Retrieval-Competition.git@rrf-exp

# Install bm25s for memory-efficient BM25 (CRITICAL for 12GB RAM)
!pip install -q bm25s
print("bm25s installed (memory-efficient BM25)")

# Install faiss-gpu with CUDA 11.8 support for Colab T4 GPU
try:
    import faiss
    print("FAISS already installed")
except ImportError:
    print("Installing faiss-gpu for CUDA 11.8...")
    !pip install -q faiss-gpu-cuda118
    print("faiss-gpu-cuda118 installed")

# Fallback: if faiss-gpu fails, install faiss-cpu (BM25 will use torch sparse on GPU)
try:
    import faiss
    if not hasattr(faiss, 'StandardGpuResources'):
        print("FAISS GPU not available, will use torch sparse for BM25")
except:
    print("Installing faiss-cpu as fallback...")
    !pip install -q faiss-cpu

!pip install -q sentence-transformers
!pip install -q -r https://raw.githubusercontent.com/samin-irtiza/Omnilex-Agentic-Retrieval-Competition/rrf-exp/requirements.txt

# Clone repo to access source code and configs
!git clone -q https://github.com/samin-irtiza/Omnilex-Agentic-Retrieval-Competition.git --branch rrf-exp --depth 1 /content/Omnilex-Agentic-Retrieval-Competition
%cd /content/Omnilex-Agentic-Retrieval-Competition

# Install the omnilex package in development mode
!pip install -q -e .

print("Setup complete!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.

In [2]:
# Import core libraries
import os
import sys
import json
from pathlib import Path
from typing import List, Tuple, Dict, Callable

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# === ADD REPO SRC TO PYTHON PATH ===
# Try to find the repo - check common locations (samin-irtiza fork)
REPO_LOCATIONS = [
    Path("/content/Omnilex-Agentic-Retrieval-Competition"),
    Path("./Omnilex-Agentic-Retrieval-Competition"),
    Path("/content/Omnilex-Agentic-Retrieval-Competition/src"),
    Path("./src"),
]

REPO_ROOT = None
for loc in REPO_LOCATIONS:
    if loc.exists():
        if loc.name == "src":
            REPO_ROOT = loc.parent
        else:
            REPO_ROOT = loc
        break

if REPO_ROOT is None:
    # Try git clone if repo not found (samin-irtiza fork, rrf-exp branch)
    print("Repo not found, cloning samin-irtiza/rrf-exp branch...")
    os.system("git clone -q https://github.com/samin-irtiza/Omnilex-Agentic-Retrieval-Competition.git --branch rrf-exp --depth 1 /content/Omnilex-Agentic-Retrieval-Competition")
    REPO_ROOT = Path("/content/Omnilex-Agentic-Retrieval-Competition")

# Add src to path
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print(f"Added {SRC_PATH} to Python path")
print(f"Repo root: {REPO_ROOT}")

# PyTorch and GPU detection
import torch
import torch.cuda as cuda

def detect_gpu():
    """Detect and configure GPU (Nvidia T4 optimized)."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB VRAM)")

        # Clear any existing cache
        torch.cuda.empty_cache()

        # Set memory fraction to avoid OOM (leave some for system)
        torch.cuda.set_per_process_memory_fraction(0.90)  # Use 90% of 15GB = ~13.5GB

        return device, True
    else:
        print("No GPU detected, using CPU")
        return torch.device("cpu"), False

DEVICE, GPU_AVAILABLE = detect_gpu()

# Import omnilex modules
from omnilex.retrieval.fusion import rrf_fusion, hybrid_search, score_normalize
# Use memory-efficient bm25s implementation (CRITICAL for 12GB RAM)
from omnilex.retrieval.bm25_index_bm25s import BM25SIndex, build_bm25s_index, search_bm25s
from omnilex.retrieval.dense_index import DenseIndex, load_jsonl_corpus as load_dense_corpus
from omnilex.citations.normalizer import CitationNormalizer
from omnilex.evaluation.scorer import evaluate_submission

# Initialize citation normalizer
citation_normalizer = CitationNormalizer()

print("Libraries imported successfully!")
print(f"Device: {DEVICE}, GPU Available: {GPU_AVAILABLE}")

Added /content/Omnilex-Agentic-Retrieval-Competition/src to Python path
Repo root: /content/Omnilex-Agentic-Retrieval-Competition
GPU detected: Tesla T4 (15.6 GB VRAM)
Libraries imported successfully!
Device: cuda, GPU Available: True


## 2. Configuration
Modify these parameters to experiment with different settings. Optimized for T4 GPU and 12GB RAM.

In [7]:
# === PATH CONFIGURATION ===
# Mount Google Drive if your datasets are stored there
from google.colab import drive
drive.mount('/content/drive')
%cp -vr '/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/' /content/Omnilex-Agentic-Retrieval-Competition
# Option 1: Upload datasets manually (run this cell and upload files)
# from google.colab import files
# uploaded = files.upload()  # Upload train.csv, val.csv, test.csv, federal_laws.jsonl, court_decisions.jsonl

# Option 2: Use provided dataset paths (update these when datasets are available)
# Use repo root from import section if available, otherwise default to content
DATA_DIR = REPO_ROOT / "data" if REPO_ROOT else Path("/content/Omnilex-Agentic-Retrieval-Competition/data")
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

# === DEVICE & BATCH CONFIGURATION (Optimized for Nvidia T4: 15GB VRAM, 12GB RAM) ===
BATCH_SIZE = 64  # Embedding batch size (adjust based on RAM: 32-64 for 12GB RAM)
EMBEDDING_BATCH_SIZE = 32  # Smaller batch for embedding generation to avoid OOM

# === RETRIEVAL CONFIGURATION ===
BM25_TOP_K = 50  # Number of results to retrieve from BM25
DENSE_TOP_K = 50  # Number of results to retrieve from Dense search
FINAL_TOP_K = 10  # Number of citations to return after fusion

# === RRF CONFIGURATION ===
RRF_K = 60  # RRF parameter (default 60, higher = more weight to lower-ranked results)

# === DENSE RETRIEVAL CONFIGURATION ===
DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # Lightweight, fast model - good for T4
# Alternative models: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2" (multilingual)

# === DATASET MODE ===
DATASET_MODE = "val"  # Options: "train", "val", "test"

# === MEMORY OPTIMIZATION ===
CACHE_INDICES = True  # Cache processed indices to avoid recomputation
CHUNK_SIZE = 1000  # Process data in chunks to avoid high RAM usage

# Create processed directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset mode: {DATASET_MODE}")
print(f"Data directory: {DATA_DIR}")
print(f"BM25 top-k: {BM25_TOP_K}, Dense top-k: {DENSE_TOP_K}, Final top-k: {FINAL_TOP_K}")
print(f"Device: {DEVICE}, Batch size: {BATCH_SIZE}, Embedding batch: {EMBEDDING_BATCH_SIZE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw' -> '/content/Omnilex-Agentic-Retrieval-Competition/data/raw'
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw/court_considerations.csv' -> '/content/Omnilex-Agentic-Retrieval-Competition/data/raw/court_considerations.csv'
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw/laws_de.csv' -> '/content/Omnilex-Agentic-Retrieval-Competition/data/raw/laws_de.csv'
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw/sample_submission.csv' -> '/content/Omnilex-Agentic-Retrieval-Competition/data/raw/sample_submission.csv'
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw/train.csv' -> '/content/Omnilex-Agentic-Retrieval-Competition/data/raw/train.csv'
'/content/drive/MyDrive/Kaggle Swiss Law Agentic Retrieval/data/raw/val.csv' -> '/c

## 3. Load Data (Memory-Efficient)

In [8]:
# Load query dataset
query_file = RAW_DATA_DIR / f"{DATASET_MODE}.csv"
if not query_file.exists():
    # Fallback to test.csv if mode file not found
    query_file = RAW_DATA_DIR / "test.csv"
    print(f"Warning: {DATASET_MODE}.csv not found, using test.csv")

assert query_file.exists(), f"Query file not found: {query_file}. Please upload datasets first."

queries_df = pd.read_csv(query_file)
print(f"Loaded {len(queries_df)} queries from {query_file}")
print(f"Columns: {list(queries_df.columns)}")
queries_df.head()

Loaded 10 queries from /content/Omnilex-Agentic-Retrieval-Competition/data/raw/val.csv
Columns: ['query_id', 'query', 'gold_citations']


,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


In [10]:
# Load legal corpora (federal laws + court decisions) - Memory Efficient
federal_laws_path = PROCESSED_DATA_DIR / "federal_laws.jsonl"
court_decisions_path = PROCESSED_DATA_DIR / "court_decisions.jsonl"

# Fallback to raw data if processed not available
if not federal_laws_path.exists():
    federal_laws_path = RAW_DATA_DIR / "federal_laws.jsonl"
if not court_decisions_path.exists():
    court_decisions_path = RAW_DATA_DIR / "court_decisions.jsonl"

assert federal_laws_path.exists(), f"Federal laws corpus not found: {federal_laws_path}"
assert court_decisions_path.exists(), f"Court decisions corpus not found: {court_decisions_path}"

def load_jsonl_chunked(filepath, chunk_size=1000):
    """Load JSONL file in chunks to avoid high RAM usage."""
    documents = []
    chunk = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f, desc=f'Loading {filepath.name}', unit='lines')):
            if line.strip():
                try:
                    chunk.append(json.loads(line))
                    if len(chunk) >= chunk_size:
                        documents.extend(chunk)
                        chunk = []
                        # Clear memory periodically
                        if GPU_AVAILABLE:
                            torch.cuda.empty_cache()
                except json.JSONDecodeError:
                    print(f'Warning: Could not parse line {i} in {filepath.name}')
        if chunk:  # Don't forget the last chunk
            documents.extend(chunk)
    return documents

# Load corpora with memory-efficient chunked loading
print("Loading federal laws...")
federal_laws = load_jsonl_chunked(federal_laws_path, chunk_size=CHUNK_SIZE)
print(f"Loaded {len(federal_laws)} federal law documents")

print("Loading court decisions...")
court_decisions = load_jsonl_chunked(court_decisions_path, chunk_size=CHUNK_SIZE)
print(f"Loaded {len(court_decisions)} court decision documents")

# Combine all documents for unified retrieval
all_documents = federal_laws + court_decisions
print(f"Total documents: {len(all_documents)}")

# Clear individual lists to save memory
federal_laws = None
court_decisions = None
import gc
gc.collect()
if GPU_AVAILABLE:
    torch.cuda.empty_cache()

# Create document ID to citation mapping
doc_id_to_citation = {}
for idx, doc in enumerate(tqdm(all_documents, desc="Mapping citations", unit="docs")):
    citation = doc.get("citation", "")
    if citation:
        doc_id_to_citation[idx] = citation

print(f"Mapped {len(doc_id_to_citation)} document citations")

Loading federal laws...


Loading federal_laws.jsonl: 0lines [00:00, ?lines/s]

Loaded 175933 federal law documents
Loading court decisions...


Loading court_decisions.jsonl: 0lines [00:00, ?lines/s]

Loaded 2476315 court decision documents
Total documents: 2652248


Mapping citations:   0%|          | 0/2652248 [00:00<?, ?docs/s]

Mapped 2652248 document citations


## 4. Build/Load Retrieval Indices

In [ ]:
# === BM25 Index (Memory-Efficient with bm25s + mmap) ===# Using bm25s library with memory-mapped sparse matrices# This keeps index on DISK (not RAM) - CRITICAL for 12GB Colab# Reduces RAM usage from ~4-10GB to ~0.5-2GBFORCE_BM25 = True  # Enable BM25 with memory-mapped indexbm25_index = None  # Default to Nonebm25_index_path = PROCESSED_DATA_DIR / "bm25s_index.pkl"bm25_mmap_path = PROCESSED_DATA_DIR / "bm25s_mmap"if FORCE_BM25:    # Try to load cached index first (with memory-mapping)    if CACHE_INDICES and bm25_index_path.exists():        try:            print(f"Loading cached BM25 index from {bm25_index_path}...")            # Load with mmap=True to keep index on disk (low RAM)            bm25_index = BM25SIndex.load(bm25_index_path, mmap=True)            print(f"BM25 index loaded successfully (memory-mapped)!")        except Exception as e:            print(f"Failed to load cached index: {e}")            bm25_index = None        # Build index if not loaded from cache    if bm25_index is None:        try:            import psutil            available_gb = psutil.virtual_memory().available / (1024**3)            print(f"Available RAM: {available_gb:.1f} GB")                        # Build BM25 index with memory-mapping (CRITICAL for 12GB RAM)            print("Building BM25 Index with bm25s (memory-efficient)...")            print(f"Using memory-mapped index to minimize RAM usage")                        # Build index with mmap support            bm25_index = build_bm25s_index(                documents=all_documents,                text_field='text',                citation_field='citation',                use_mmap=True,  # CRITICAL: Keep index on disk, not RAM                mmap_path=str(bm25_mmap_path),  # Path for mmap files                vocab_size=50000,  # Limit vocab to control memory            )                        print(f"BM25 index built successfully!")            print(f"Index size: {len(all_documents)} documents")                        # Save index to cache (with mmap support)            if CACHE_INDICES:                try:                    bm25_index.save(bm25_index_path)                    print(f"BM25 index cached to {bm25_index_path}")                    print(f"Mmap files saved to {bm25_mmap_path}")                except Exception as e:                    print(f"Warning: Could not cache BM25 index: {e}")                    except Exception as e:            print(f"BM25 build failed: {e}")            print("Falling back to Dense-only retrieval.")            bm25_index = Noneelse:    print("BM25 disabled.")    print("Using Dense-only retrieval.")print(f"BM25 index: {'ENABLED' if bm25_index else 'DISABLED'}")# === Test BM25 Index ===if bm25_index is not None:    print("\nTesting BM25 index with sample query...")    test_query = "court order"    try:        # Use search_with_scores for (index, score) tuples
        test_results = bm25_index.search_with_scores(test_query, top_k=5)
        print(f"Test query: '{test_query}'")        print(f"Found {len(test_results)} results")        if test_results:            print("Top 3 results:")            for i, (doc_idx, score) in enumerate(test_results[:3]):
                citation = bm25_index.doc_citations[doc_idx] if doc_idx < len(bm25_index.doc_citations) else 'N/A'
        print("BM25 index is working correctly!")    except Exception as e:        print(f"BM25 test failed: {e}")        print("Will proceed with Dense-only retrieval.")        bm25_index = None

Building BM25 index...


In [ ]:
# === Dense Index (Semantic Search) - GPU Optimized ===
dense_index_path = PROCESSED_DATA_DIR / "combined_dense_index"

if CACHE_INDICES and dense_index_path.with_suffix('.index').exists():
    print(f"Loading Dense index from {dense_index_path}")
    dense_index = DenseIndex.load(dense_index_path)
    # Move to GPU if available
    if GPU_AVAILABLE and hasattr(dense_index, 'model') and dense_index.model is not None:
        try:
            dense_index.model.to(DEVICE)
            print(f"Moved dense model to {DEVICE}")
        except Exception as e:
            print(f"Warning: Could not move model to GPU: {e}")
else:
    print("Building Dense index with GPU acceleration...")
    print(f"Using model: {DENSE_MODEL_NAME}")

    try:
        dense_index = DenseIndex(
            documents=all_documents,
            model_name=DENSE_MODEL_NAME,
        )

        # Build with progress tracking
        dense_index.build(all_documents, model_name=DENSE_MODEL_NAME)

        # Move model to GPU after loading
        if GPU_AVAILABLE and hasattr(dense_index, 'model') and dense_index.model is not None:
            dense_index.model.to(DEVICE)
            print(f"Model moved to GPU: {DEVICE}")

        if CACHE_INDICES:
            dense_index.save(dense_index_path)
            print(f"Dense index saved to {dense_index_path}")

    except Exception as e:
        print(f"Error building dense index: {e}")
        if GPU_AVAILABLE:
            torch.cuda.empty_cache()

print(f"Dense index ready with {len(dense_index.documents)} documents")

# Clear GPU cache after building index
if GPU_AVAILABLE:
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## 4.5 GPU Memory Cleanup & Optimization
Clear GPU memory after building indices and optimize for inference.

In [ ]:
# === GPU Memory Cleanup after Index Building ===
def cleanup_gpu_memory():
    """Clean up GPU memory and report status."""
    if GPU_AVAILABLE:
        # Clear CUDA cache
        torch.cuda.empty_cache()

        # Force garbage collection
        import gc
        gc.collect()

        # Report memory status
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU memory allocated: {allocated:.2f} GB")
        print(f"GPU memory reserved: {reserved:.2f} GB")
        print(f"GPU memory free: {total - reserved:.2f} GB")

        # Move model to eval mode and optimize for inference
        if hasattr(dense_index, 'model') and dense_index.model is not None:
            dense_index.model.eval()  # Set to eval mode for inference
            print("Dense model set to eval mode")
    else:
        print("No GPU to clean up")

    # Clear all_documents from memory after indices are built (if not needed)
    global all_documents
    if 'all_documents' in globals():
        print(f"Clearing all_documents from memory ({len(all_documents)} docs)")
        all_documents = None
        import gc
        gc.collect()

# Run cleanup
cleanup_gpu_memory()

print("\nMemory cleanup complete! Ready for inference.")

## 5. RAG Pipeline with RRF Fusion
This section defines the core retrieval + fusion logic. Extend by adding more retrievers.

In [ ]:
def retrieve_with_bm25(query: str, top_k: int = BM25_TOP_K) -> List[Tuple[int, float]]:
    """Retrieve documents using BM25 keyword search with memory-mapped index.
    
    Uses bm25s library with memory-mapped sparse matrices.
    Memory-efficient: keeps index on disk, minimizes RAM usage.
    
    Args:
        query: Search query string
        top_k: Number of results to return
    
    Returns:
        List of (document_index, score) tuples
    """
    if bm25_index is None:
        print("WARNING: BM25 index not available. Returning empty results.")
        return []
    
    try:
        # Use bm25s search_with_scores for (index, score) tuples
        results = bm25_index.search_with_scores(query, top_k=top_k)
        return results
    except Exception as e:
        print(f"BM25 search error: {e}")
        return []
    finally:
        # Clear GPU cache after search to free VRAM
        if GPU_AVAILABLE:
            import torch
            torch.cuda.empty_cache()
    

def retrieve_with_dense(query: str, top_k: int = DENSE_TOP_K) -> List[Tuple[int, float]]:
    """Retrieve documents using Dense semantic search.

    Args:
        query: Search query string
        top_k: Number of results to return

    Returns:
        List of (document_index, score) tuples
    """
    # Use torch.no_grad() for memory-efficient inference
    if GPU_AVAILABLE and hasattr(dense_index, "model"):
        with torch.no_grad():
            return dense_index.search(query, top_k=top_k)
    else:
        return dense_index.search(query, top_k=top_k)


# === EXTEND HERE: Add more retrievers ===
# Example: TF-IDF, custom embedding models, etc.
# def retrieve_with_tfidf(query: str, top_k: int) -> List[Tuple[int, float]]:
#     ...


def rrf_fusion_pipeline(
    query: str,
    retrievers: List[Callable],
    top_k: int = FINAL_TOP_K,
    rrf_k: int = RRF_K,
) -> List[str]:
    """Run multiple retrievers, fuse results with RRF, return top citations.

    Args:
        query: Search query
        retrievers: List of retriever functions (each returns List[Tuple[int, float]])
        top_k: Number of final citations to return
        rrf_k: RRF parameter

    Returns:
        List of normalized citation strings
    """
    # Get results from all retrievers
    result_lists = [retriever(query) for retriever in retrievers]

    # Normalize scores for each retriever (rank-based normalization)
    normalized_lists = [score_normalize(results, method="rank") for results in result_lists]

    # Fuse with RRF
    fused_results = rrf_fusion(normalized_lists, k=rrf_k)

    # Get top-k document IDs and map to citations
    top_doc_ids = [doc_id for doc_id, _ in fused_results[:top_k]]
    citations = []
    seen = set()

    for doc_id in top_doc_ids:
        if doc_id in doc_id_to_citation:
            citation = citation_normalizer.normalize(doc_id_to_citation[doc_id])
            # Fallback to raw citation if normalization fails
            if not citation:
                citation = doc_id_to_citation[doc_id]
            if citation and citation not in seen:
                seen.add(citation)
                citations.append(citation)

    return citations


# Define active retrievers (BM25 is optional based on memory)
ACTIVE_RETRIEVERS = []
if bm25_index is not None:
    ACTIVE_RETRIEVERS.append(retrieve_with_bm25)
    print("BM25 retriever: ENABLED")
else:
    print("BM25 retriever: DISABLED (low memory or skipped)")

ACTIVE_RETRIEVERS.append(retrieve_with_dense)
print(f"Active retrievers: {len(ACTIVE_RETRIEVERS)}")


In [ ]:
# Test the pipeline with a sample query
test_query = "What are the requirements for a valid contract under Swiss law?"
print(f"Test query: {test_query}")

sample_citations = rrf_fusion_pipeline(
    query=test_query,
    retrievers=ACTIVE_RETRIEVERS,
    top_k=5,
)

print("\nTop 5 citations from RRF fusion:")
for i, cit in enumerate(sample_citations, 1):
    print(f"{i}. {cit}")

## 6. Process All Queries & Generate Predictions

In [ ]:
predictions = []

for idx, row in tqdm(enumerate(queries_df.iterrows()), total=len(queries_df), desc="Processing queries"):
    _, row = row  # Unpack enumerate result
    query_id = row["query_id"]
    query_text = row["query"]

    # Retrieve and fuse citations
    citations = rrf_fusion_pipeline(
        query=query_text,
        retrievers=ACTIVE_RETRIEVERS,
        top_k=FINAL_TOP_K,
    )

    # Format as semicolon-separated string (use " " if no citations)
    predicted_citations = ";".join(citations) if citations else " "

    predictions.append({
        "query_id": query_id,
        "predicted_citations": predicted_citations,
    })

    # Periodic memory cleanup during batch processing
    if GPU_AVAILABLE and idx % 100 == 0:
        torch.cuda.empty_cache()

predictions_df = pd.DataFrame(predictions)
print(f"\nGenerated predictions for {len(predictions_df)} queries")
predictions_df.head(10)

## 7. Evaluation (Validation Set Only)
Runs only if gold citations are available in the dataset.

In [ ]:
if "gold_citations" in queries_df.columns:
    print("Gold citations found - running evaluation...")

    # Compute metrics using omnilex evaluation module
    metrics = evaluate_submission(
        submission_df=predictions_df,
        gold_df=queries_df,
    )

    print("\n=== Evaluation Results ===")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("No gold citations found - skipping evaluation. Use val.csv for validation.")

## 8. Create Submission File

In [ ]:
# Save submission in Kaggle format
submission_path = Path("/content/submission.csv")
predictions_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")

# Preview submission
print("\nSample submission rows:")
print(predictions_df.head())

# Download submission (Colab only)
# from google.colab import files
# files.download(submission_path)

## 9. Extendability Guide
To improve the pipeline, modify these sections:

### Add New Retrievers
1. Define a new retriever function in Section 5 that returns `List[Tuple[int, float]]`
2. Add the function to `ACTIVE_RETRIEVERS`

### Modify Fusion Logic
- Adjust `RRF_K` parameter in Section 2
- Replace RRF with `convex_fusion` or `hybrid_search` from `omnilex.retrieval.fusion`
- Add score normalization methods (minmax, zscore)

### Integrate LLM Reranking
- Add an LLM reranking step after RRF fusion to select the most relevant citations
- Use the existing `omnilex.llm` module for local LLM loading

### Improve Citation Normalization
- Modify the `CitationNormalizer` class from `omnilex.citations.normalizer`
- Add custom regex patterns for edge cases
- Use `citation_normalizer.canonicalize_list()` for batch normalization

### Swap Embedding Models
- Change `DENSE_MODEL_NAME` in Section 2 to a different sentence-transformer model
- For multilingual support, use `paraphrase-multilingual-MiniLM-L12-v2`

### Hardware Optimization Notes
- **GPU**: Sentence-transformers automatically uses CUDA if available
- **Memory**: Chunked loading and periodic cache clearing prevent OOM
- **FAISS**: Uses CPU version (faiss-cpu) - GPU acceleration via torch for embeddings